# 情報数学Ⅲ 第13回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

In [ ]:
# 使用するライブラリのインポート
import pandas as pd
import statsmodels.formula.api as smf # モデルの記述を文字列の数列（formmula）の形で行える
import statsmodels.api as sm # 今回は使用しない．書き方が少し異なる
from sklearn.preprocessing import StandardScaler

# 日本語表示: Colab環境用
#!pip install japanize_matplotlib
#import japanize_matplotlib

# 日本語表示: 教員確認用（Mac）
# 日本語フォントを指定（ヒラギノ角ゴがmacOSに標準搭載）
#plt.rcParams['font.family'] = 'Hiragino Sans'


## 例題1：重回帰分析：地価に影響する土地の種類は？

* 地価（土地の価格）が周辺環境からどのような影響を受けているのかを調べてみよう
* ノートブックにそのデータが入っている
* 例えば，番号1の地点の地価は，319千円/$m^2$で，その周辺25万$m^2$（500m四方）には，農地が200$m^2$，林地が400$m^2$，商業地が16000$m^2$，密集住宅が7200$m^2$存在していることを示す
* ざっと見た感じでは，ちょっと傾向はわからないが・・・


In [ ]:
data = [
    {'番号': 1, '地価': 319, '農地面積': 2, '林地面積': 4, '商業地面積': 164, '密集住宅面積': 72},
    {'番号': 2, '地価': 261, '農地面積': 20, '林地面積': 3, '商業地面積': 288, '密集住宅面積': 103},
    {'番号': 3, '地価': 279, '農地面積': 19, '林地面積': 1, '商業地面積': 195, '密集住宅面積': 190},
    {'番号': 4, '地価': 249, '農地面積': 52, '林地面積': 7, '商業地面積': 225, '密集住宅面積': 127},
    {'番号': 5, '地価': 235, '農地面積': 21, '林地面積': 0, '商業地面積': 328, '密集住宅面積': 304},
    {'番号': 6, '地価': 273, '農地面積': 0, '林地面積': 0, '商業地面積': 637, '密集住宅面積': 100},
    {'番号': 7, '地価': 264, '農地面積': 8, '林地面積': 2, '商業地面積': 253, '密集住宅面積': 242},
    {'番号': 8, '地価': 212, '農地面積': 149, '林地面積': 83, '商業地面積': 51, '密集住宅面積': 23},
    {'番号': 9, '地価': 255, '農地面積': 52, '林地面積': 26, '商業地面積': 106, '密集住宅面積': 84},
    {'番号':10, '地価': 310, '農地面積': 99, '林地面積': 233, '商業地面積': 104, '密集住宅面積': 168},
    {'番号':11, '地価': 306, '農地面積': 68, '林地面積': 327, '商業地面積': 59, '密集住宅面積': 191},
    {'番号':12, '地価': 296, '農地面積': 136, '林地面積': 342, '商業地面積': 116, '密集住宅面積': 144},
    {'番号':13, '地価': 266, '農地面積': 19, '林地面積': 0, '商業地面積': 242, '密集住宅面積': 266},
    {'番号':14, '地価': 346, '農地面積': 9, '林地面積': 13, '商業地面積': 98, '密集住宅面積': 97},
    {'番号':15, '地価': 284, '農地面積': 51, '林地面積': 65, '商業地面積': 98, '密集住宅面積': 116},
    {'番号':16, '地価': 384, '農地面積': 1, '林地面積': 444, '商業地面積': 21, '密集住宅面積': 87},
    {'番号':17, '地価': 331, '農地面積': 32, '林地面積': 151, '商業地面積': 189, '密集住宅面積': 26},
    {'番号':18, '地価': 339, '農地面積': 6, '林地面積': 10, '商業地面積': 85, '密集住宅面積': 60},
    {'番号':19, '地価': 344, '農地面積': 7, '林地面積': 35, '商業地面積': 87, '密集住宅面積': 77},
    {'番号':20, '地価': 272, '農地面積': 31, '林地面積': 0, '商業地面積': 383, '密集住宅面積': 118},
    {'番号':21, '地価': 364, '農地面積': 7, '林地面積': 180, '商業地面積': 77, '密集住宅面積': 130},
    {'番号':22, '地価': 263, '農地面積': 13, '林地面積': 7, '商業地面積': 194, '密集住宅面積': 97},
    {'番号':23, '地価': 340, '農地面積': 35, '林地面積': 110, '商業地面積': 70, '密集住宅面積': 103},
    {'番号':24, '地価': 301, '農地面積': 10, '林地面積': 87, '商業地面積': 85, '密集住宅面積': 171},
    {'番号':25, '地価': 390, '農地面積': 3, '林地面積': 0, '商業地面積': 198, '密集住宅面積': 50},
    {'番号':26, '地価': 316, '農地面積': 62, '林地面積': 88, '商業地面積': 100, '密集住宅面積': 39},
    {'番号':27, '地価': 295, '農地面積': 77, '林地面積': 68, '商業地面積': 139, '密集住宅面積': 229},
    {'番号':28, '地価': 382, '農地面積': 0, '林地面積': 139, '商業地面積': 15, '密集住宅面積': 52},
    {'番号':29, '地価': 361, '農地面積': 3, '林地面積': 4, '商業地面積': 191, '密集住宅面積': 57},
    {'番号':30, '地価': 370, '農地面積': 0, '林地面積': 139, '商業地面積': 18, '密集住宅面積': 0},
]

### データフレームにする


In [ ]:
df = pd.DataFrame(data)
df.head()

### 重回帰分析の実施

In [ ]:
# 重回帰分析：地価を目的変数、4つの面積を説明変数
model = smf.ols(formula='地価 ~ 農地面積 + 林地面積 + 商業地面積 + 密集住宅面積', data=df).fit()

# 結果表示
print(model.summary())

### 標準化する

In [ ]:
# 説明変数だけ抽出
features = ['農地面積', '林地面積', '商業地面積', '密集住宅面積']

# 標準化
scaler = StandardScaler()

df['農地面積_std'] = scaler.fit_transform(df[['農地面積']])
df['林地面積_std'] = scaler.fit_transform(df[['林地面積']])
df['商業地面積_std'] = scaler.fit_transform(df[['商業地面積']])
df['密集住宅面積_std'] = scaler.fit_transform(df[['密集住宅面積']])

# 標準化された変数で回帰モデルを構築
formula = '地価 ~ 農地面積_std + 林地面積_std + 商業地面積_std + 密集住宅面積_std'
model_std = smf.ols(formula=formula, data=df).fit()

# 結果出力
print(model_std.summary())

## 演習：直売所の売上に影響する変数は？

次のデータは、ある地域にある 8 つの農産物直売所についての（仮想）データである。売上（千万円/年）を目的変数とし、以下の説明変数を用いて重回帰分析を行う。
* 店舗面積（㎡）
* 駐車可能台数（台）
* フリーマーケットの有無（あり＝1、なし＝0）

### (1) 売上に最も影響を与えている要因を比較するために、店舗面積および駐車可能台数を標準化したうえで、重回帰分析を行え。分析にはフリーマーケットの有無も含めてよい。

In [ ]:
data = [
    {'直売所': 'A', '売上': 8,   '店舗面積': 100, '駐車可能台数': 250, 'フリーマーケット': 1},
    {'直売所': 'B', '売上': 10,  '店舗面積': 180, '駐車可能台数': 600, 'フリーマーケット': 1},
    {'直売所': 'C', '売上': 0.5, '店舗面積': 60,  '駐車可能台数': 100, 'フリーマーケット': 0},
    {'直売所': 'D', '売上': 1.5, '店舗面積': 80,  '駐車可能台数': 50,  'フリーマーケット': 0},
    {'直売所': 'E', '売上': 9,   '店舗面積': 190, '駐車可能台数': 120, 'フリーマーケット': 1},
    {'直売所': 'F', '売上': 13,  '店舗面積': 210, '駐車可能台数': 1000,'フリーマーケット': 1},
    {'直売所': 'G', '売上': 0.1, '店舗面積': 50,  '駐車可能台数': 50,  'フリーマーケット': 0},
    {'直売所': 'H', '売上': 13,  '店舗面積': 250, '駐車可能台数': 800, 'フリーマーケット': 1},
]

### データフレームにする

### 重回帰分析の実施

### (2) (1) の分析結果をもとにして、売上に最も強い影響を与えていると考えられる説明変数を選べ。
ただし、ここでの「強い影響」は、標準化された回帰係数の大きさに基づいて判断するものとする。

### (3) (1) の分析結果において、各説明変数の偏回帰係数に対するt検定の p値を用いて、以下の問いに答えよ。
* 有意水準を 5% に設定すると、3 つすべての説明変数において偏回帰係数は 0 でないと判断される（帰無仮説が棄却される）。
* 一方、有意水準を 1% に設定した場合は、1 つの説明変数のみが有意と判断される。

このとき、有意水準 1% でも有意となる説明変数はどれか。